In [ ]:
import numpy as np
import torch
import torch.optim as optim
import matplotlib.pyplot as plt
from pathlib import Path
import copy
import h5py
# print(Path.cwd()) # Check current path

### Configurations, Probabilities, and Moments

### Graph Topology

Generate Graph Edges

In [ ]:
def generate_lattice_graph(rows, cols):
    """
    Generates edges as tuples for a lattice graph.
    """
    edges = []

    for r in range(rows):
        for c in range(cols):
            node = r * cols + c

            # Connect to right neighbor
            if c < cols - 1:
                right = node + 1
                edges.append((node, right))

            # Connect to bottom neighbor
            if r < rows - 1:
                below = node + cols
                edges.append((node, below))

    return edges

Generate Sequences

In [ ]:
def lattice_skip_sequence(n):
    assert n >= 2, "Use n >= 2"

    seq = []

    # Pass 1
    for r in range(n):
        startc = 0 if r % 2 == 0 else 1
        for c in range(startc, n, 2):
            seq.append(r * n + c)

    # Pass 2
    for r in range(n):
        startc = 1 if r % 2 == 0 else 0
        for c in range(startc, n, 2):
            seq.append(r * n + c)

    return seq


def idx(r, c, n):
    return r * n + c

def lattice_diagonal_sequence(n):
    assert n >= 2, "Use n >= 2"

    # Build diagonals keyed by d = c - r
    diags = {}
    for d in range(-(n - 1), n):
        v = []

        rmin = max(0, -d)
        rmax = min(n - 1, n - 1 - d)

        for r in range(rmin, rmax + 1):
            c = r + d
            v.append(idx(r, c, n))

        diags[d] = v

    # ---- Main diagonal: center, then skip-one outward (LEFT FIRST) ----
    main = diags[0]

    # left-of-center if even
    mid = (n - 1) // 2

    picked = [main[mid]]

    for step in range(2, n, 2):
        l = mid - step
        r = mid + step

        if l >= 0:
            picked.append(main[l])
        if r < n:
            picked.append(main[r])

    # Append remaining main-diagonal nodes in natural order
    for x in main:
        if x not in picked:
            picked.append(x)

    seq = picked.copy()

    # ---- Upper triangle (skip one diagonal, pick next) ----
    for d in range(2, n, 2):
        seq.extend(diags[d])

    for d in range(1, n, 2):
        seq.extend(diags[d])

    # ---- Lower triangle (skip one diagonal, pick next) ----
    for d in range(-2, -n, -2):
        seq.extend(diags[d])

    for d in range(-1, -n, -2):
        seq.extend(diags[d])

    return seq


def parent_count_if_next(adj, selected, candidate):
    """
    Number of autoregressive parents candidate would have if selected next.

    selected[j] is True for already selected nodes.
    The parents are selected nodes adjacent to the connected component
    containing candidate in the subgraph induced by unselected nodes.
    """
    n = len(adj)

    # Find candidate's connected component among currently unselected nodes.
    component = set()
    stack = [candidate]

    while stack:
        node = stack.pop()

        if node in component:
            continue

        component.add(node)

        for neighbor in adj[node]:
            if not selected[neighbor] and neighbor not in component:
                stack.append(neighbor)

    # Selected boundary nodes are the effective parents.
    parents = set()

    for node in component:
        for neighbor in adj[node]:
            if selected[neighbor]:
                parents.add(neighbor)

    return len(parents)


def greedy_frontier_sequence(edges, n):
    """
    Greedy autoregressive ordering with one-step lookahead.
    """

    # Build adjacency list.
    adj = [[] for _ in range(n)]

    for u, v in edges:
        adj[u].append(v)
        adj[v].append(u)

    sequence = []
    selected = [False] * n
    realized_sizes = []

    while len(sequence) < n:
        best_node = None
        best_current_size = None
        best_score = None

        for candidate in range(n):
            if selected[candidate]:
                continue

            # Parent count if candidate is selected now.
            current_size = parent_count_if_next(adj, selected, candidate)

            # Temporarily select candidate.
            selected[candidate] = True

            prospective_sizes = []

            for other in range(n):
                if not selected[other]:
                    prospective_sizes.append(parent_count_if_next(adj, selected, other))

            selected[candidate] = False

            realized_max = max(realized_sizes, default=0)
            prospective_max = max(prospective_sizes, default=0)

            projected_max = max(realized_max, current_size, prospective_max)

            projected_count = (
                sum(size == projected_max for size in realized_sizes)
                + int(current_size == projected_max)
                + sum(size == projected_max for size in prospective_sizes)
            )

            future_max = prospective_max

            future_count = sum(size == future_max for size in prospective_sizes)

            # Same lexicographic priorities as the Julia version.
            score = (projected_max, projected_count, future_max, future_count, sum(prospective_sizes), current_size, candidate)

            if best_score is None or score < best_score:
                best_score = score
                best_node = candidate
                best_current_size = current_size

        sequence.append(best_node)
        realized_sizes.append(best_current_size)
        selected[best_node] = True

    return sequence

Defining Parent Set

In [ ]:
def build_autoregressive_parents(edges, sequence):
    """
    Returns a dictionary of nodes and parent nodes, for an autoregressive model.

    This algorithm visits each node sequentially as per the input sequence, 
    node j is denoted as a parent node to i if there exists a path  
    (i,k1),(k1,k2),...,(km,j), with each kl=sequence[i-1], connecting i to j, and
    kl !∊ parent(i). 
    """

    # Build adjacency list
    adj = {}
    for u, v in edges:
        adj.setdefault(u, []).append(v)
        adj.setdefault(v, []).append(u)

    parent_dict = {}

    def has_valid_path(parent, child, blocked):
        visited = set()
        stack = [parent]

        while stack:
            node = stack.pop()

            if node == child:
                return True

            visited.add(node)

            for neighbor in adj[node]:
                if neighbor not in visited and neighbor not in blocked:
                    stack.append(neighbor)

        return False

    for i, node in enumerate(sequence):
        previous_nodes = sequence[:i]
        parents = []

        for candidate in previous_nodes:
            blocked = set(previous_nodes)
            blocked.remove(candidate)  # Only allow the candidate to walk through

            if has_valid_path(candidate, node, blocked):
                parents.append(candidate)

        parent_dict[node] = parents

    return parent_dict

In [ ]:
def compress_samples(samples):
    unique_configs, counts = np.unique(samples, axis=0, return_counts=True)
    compressed = np.hstack((counts[:, None], unique_configs))

    return compressed.astype(np.int64)

### neuRISE Learning - GLOBAL
Parameterize all conditionals by one neural network

In [ ]:
def make_mlp(input_dim, hidden_dim, output_dim=2, num_hidden_layers=2):
    layers = []
    d = input_dim

    for _ in range(num_hidden_layers):
        layers.append(torch.nn.Linear(d, hidden_dim))
        layers.append(torch.nn.ReLU())
        d = hidden_dim

    layers.append(torch.nn.Linear(d, output_dim))
    return torch.nn.Sequential(*layers)


In [ ]:
def neuRISE_loss_global(O, samples, parents_device, mlp, q=2, 
                        node_batch_size=None, rng=None):
    
    device = next(mlp.parameters()).device
    samples = torch.as_tensor(samples, dtype=torch.float32, device=device)
    counts = samples[:, 0]
    spins = samples[:, 1:]

    n_obs = samples.shape[0]
    n_nodes = len(O)

    if rng is None:
        rng = np.random.default_rng()

    if node_batch_size is None or node_batch_size == n_nodes:
        pos_batch = np.arange(n_nodes)
    else:
        pos_batch = rng.choice(n_nodes, size=node_batch_size, replace=False)

    weights = counts / counts.sum()

    node_losses = []

    for pos in pos_batch:
        node = int(O[pos])

        sigma_i = spins[:, node]
        parents = parents_device[node]
        
        sigma_masked = torch.zeros((n_obs, n_nodes), dtype=torch.float32, device=device)
        parent_mask = torch.zeros((n_obs, n_nodes), dtype=torch.float32, device=device)

        if len(parents) > 0:
            sigma_masked[:, parents] = spins[:, parents]
            parent_mask[:, parents] = 1.0

        node_onehot = torch.zeros((n_obs, n_nodes), dtype=torch.float32, device=device)
        node_onehot[:, node] = 1.0

        nn_output = mlp(torch.cat([sigma_masked, parent_mask, node_onehot], dim=1))
        phi_minus = torch.where(sigma_i == -1, 1.0 - 1.0 / q, -1.0 / q)
        phi_plus = torch.where(sigma_i == 1, 1.0 - 1.0 / q, -1.0 / q)

        node_losses.append(torch.sum(weights * torch.exp(
                    -phi_minus * nn_output[:, 0] - phi_plus * nn_output[:, 1]
                )))

    return torch.stack(node_losses).mean()

In [ ]:
def neuRISE_global(n, O, parents_dict, samples, hd, od, ep, lr,
                   num_hidden_layers=2, node_batch_size=None, patience=5,
                   tol=1e-5, check_every=100, print_every=500, rng=None):

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU is not available.")
    device = torch.device("cuda")
    print("Training on:", torch.cuda.get_device_name(0))

    parents_device = {
        int(node): torch.as_tensor(parents_dict[int(node)], dtype=torch.long, device=device)
        for node in O
    }
    
    samples = torch.as_tensor(samples, dtype=torch.float32, device=device)

    if rng is None:
        rng = np.random.default_rng()

    mlp = make_mlp(3 * n, hd, od, num_hidden_layers).to(device)
    
    optimizer = optim.Adam(mlp.parameters(), lr=lr)

    initial_loss = neuRISE_loss_global(
        O, samples, parents_device, mlp, q=2,
        node_batch_size=None, rng=rng).item()
    print(f"initial full loss={initial_loss:.6f}")

    best_loss = initial_loss
    best_state = copy.deepcopy(mlp.state_dict())
    stale_checks = 0

    for epoch in range(ep):
        optimizer.zero_grad()

        loss = neuRISE_loss_global(
            O, samples, parents_device, mlp, q=2,
            node_batch_size=node_batch_size, rng=rng,
        )

        loss.backward()

        should_check = (epoch + 1) % check_every == 0 or epoch == ep - 1

        if should_check:
            if node_batch_size is None:
                monitor_loss = loss.item()
            else:
                with torch.no_grad():
                    monitor_loss = neuRISE_loss_global(
                        O, samples, parents_device, mlp, q=2,
                        node_batch_size=None, rng=rng,
                    ).item()

            if best_loss - monitor_loss > tol:
                best_loss = monitor_loss
                best_state = copy.deepcopy(mlp.state_dict())
                stale_checks = 0
            else:
                stale_checks += 1

            if (epoch + 1) % print_every == 0 or epoch == ep - 1:
                print(
                    f"epoch {epoch + 1:5d}/{ep} | "
                    f"batch loss={loss.item():.6f} | "
                    f"full loss={monitor_loss:.6f} | best={best_loss:.6f}"
                )

            if stale_checks >= patience:
                break

        optimizer.step()

    mlp.load_state_dict(best_state)
    return mlp

## Main--Learning and Sampling

In [ ]:
# --- Number of vertices ---
N = 100
# Number of vertices along each edge of the lattice
L = int(np.sqrt(N))

beta = 0.2
beta_tag = str(beta).replace(".", "")

# --- Range of data and generated samples ---
Ml_range = [500,1000, 5000, 10_000]
I = len(Ml_range)

# --- Number of iterations ---
T = 10

# --- Define graph topology ---
edges = generate_lattice_graph(L, L)

ordering_keys = ("sequential", "diagonal", "greedy") #"checkerboard", 

# --- Define sequences (nodes numbering start from 0) ---
seq = np.arange(N)
# seq_skip = lattice_skip_sequence(L)
seq_diag = lattice_diagonal_sequence(L)
seq_greedy = greedy_frontier_sequence(edges, N)

models = {
    "sequential": {
        "label": "Sequential",
        "seq": seq,
        "parents": build_autoregressive_parents(edges, seq),
    },
    "diagonal": {
        "label": "Diagonal",
        "seq": seq_diag,
        "parents": build_autoregressive_parents(edges, seq_diag),
    },
    "greedy": {
        "label": "Greedy",
        "seq": seq_greedy,
        "parents": build_autoregressive_parents(edges, seq_greedy),
    },
}

# ---- Store solutions and results ---
solutions = {
    ml: {
        key: [None] * T
        for key in ordering_keys
    }
    for ml in Ml_range
}

# --- Load moment and samples ---
samples_path = Path(f"Data/10x10/Samples_10x10_M=200K_Ferro_beta={beta_tag}.csv")
data_samples = np.loadtxt(samples_path, delimiter=",").astype(np.int8)

# --- Reserve training and held-out samples ----
training_pool = data_samples[:-20_000]

# --- Neural Network Specifications ---
hidden_dim, num_hidden_layers, output_dim = 100, 2, 2
num_epochs, learning_rate = 2000, 1e-3
node_batch_size = None

# ---- Save Path ----
save_path = Path(f"Results/10x10/Results_10x10_Ferro_{beta_tag}_500.pt")
save_path.parent.mkdir(parents=True, exist_ok=True)

# ---- Main Loop ---
for j, ml in enumerate(Ml_range):

    if ml > len(training_pool):
        raise ValueError(
            f"Ml={ml} exceeds training-pool size {len(training_pool)}."
        )

    for t in range(T):
        seed = j * T + t + 100
        data_rng = np.random.default_rng(seed)

        train_indices = data_rng.choice(
            len(training_pool),
            size=ml,
            replace=False,
        )
        train_samples = compress_samples(
            training_pool[train_indices]
        )

        for key in ordering_keys:
            model = models[key]

            torch.manual_seed(seed)
            optimization_rng = np.random.default_rng(seed)

            network_global = neuRISE_global(
                N,
                model["seq"],
                model["parents"],
                train_samples,
                hidden_dim,
                output_dim,
                num_epochs,
                learning_rate,
                num_hidden_layers=num_hidden_layers,
                node_batch_size=node_batch_size,
                rng=optimization_rng,
            )

            network_global.eval()

            solutions[ml][key][t] = {
                "state_dict": {
                    name: parameter.detach().cpu().clone()
                    for name, parameter
                    in network_global.state_dict().items()
                },
                "seed": seed,
                "train_indices": train_indices.copy(),
            }

    # ---------- Checkpoint after each Ml ----------

    torch.save(
        {
            "beta": beta,
            "ordering_keys": ordering_keys,
            "solutions": solutions,
            "network_specifications": {
                "input_dim": 3 * N,
                "hidden_dim": hidden_dim,
                "output_dim": output_dim,
                "num_hidden_layers": num_hidden_layers,
                "num_epochs": num_epochs,
                "learning_rate": learning_rate,
                "node_batch_size": node_batch_size,
            },
        },
        save_path,
    )

    print(f"Saved results through Ml={ml} to {save_path}")

In [ ]:
# # To load neurise solution:
# checkpoint = torch.load("Results/neurise_global_beta_02.pt", map_location="cpu")

# ml = 5000
# key = "greedy"
# t = 2

# solution = checkpoint["solutions"][ml][key][t]

# network_global = make_mlp(
#     3 * N, hidden_dim, output_dim, num_hidden_layers
# )

# network_global.load_state_dict(solution["state_dict"])
# network_global.eval()

# gen_samples = generate_test_samples_global(
#     models[key]["seq"],
#     models[key]["parents"],
#     network_global,
#     Ms,
#     N,
#     rng=np.random.default_rng(123),
# )